In [ ]:
# -*- coding: utf-8 -*-
"""
Model comparison on the WDP dataset
===================================

Fine-tunes nine pre-trained vision transformers (supervised ViT and DeiT,
self-supervised DINOv2, vision-language CLIP) for website aesthetic score
regression and compares them on a single stratified 68/12/20
train/validation/test split of the Web Design Prototypicality dataset.

Each model goes through the three-phase protocol described in the paper:
the regression head is trained first, then the last four encoder blocks
with layer-wise learning rate decay, then the whole network at a reduced
rate. Metrics are Pearson r, Spearman rho and RMSE with bootstrap
confidence intervals, plus per-category breakdowns. Per-model test
predictions are written to CSV, which is what the later statistical
comparison of paradigms is based on.

Source of Table 3 and Figs. 1-2 in the paper.

Written for Google Colab on a single GPU. Expects the dataset archive at
/content/drive/MyDrive/datasets/webdesignprototypicality.zip.

Dependencies:
pip install transformers timm accelerate scipy pandas scikit-learn pillow tqdm matplotlib psutil
"""

# ============================================================
# 0) Environment Setup
# ============================================================
import os
import math
import random
import time
import warnings
import json
from dataclasses import dataclass, field
from typing import Tuple, List, Optional, Dict, Any
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torch import amp

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib.pyplot as plt

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

# ============================================================
# 1) Reproducibility & Device Setup
# ============================================================
def set_seed(seed: int = 42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
pin_memory = use_cuda

# Enable TF32 for performance (Ampere+ GPUs)
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print(f"Using device: {device}")
if use_cuda:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# ============================================================
# 2) Model Registry with Architecture-Specific Settings
# ============================================================
MODEL_REGISTRY = {
    # Supervised ViT (ImageNet-21k + ImageNet-1k fine-tuned)
    "vit-base-224": {
        "hf_name": "google/vit-base-patch16-224",
        "img_size": 224,
        "patch_size": 16,
        "base_lr": 3e-5,
        "head_lr": 1e-3,
        "batch_size": 16,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.85,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "description": "ViT-B/16 baseline at 224px"
    },
    "vit-base-384": {
        "hf_name": "google/vit-base-patch16-384",
        "img_size": 384,
        "patch_size": 16,
        "base_lr": 3e-5,
        "head_lr": 1e-3,
        "batch_size": 8,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.85,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "description": "ViT-B/16 at higher resolution (384px)"
    },
    "vit-large-384": {
        "hf_name": "google/vit-large-patch16-384",
        "img_size": 384,
        "patch_size": 16,
        "base_lr": 2e-5,
        "head_lr": 5e-4,
        "batch_size": 4,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.90,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "description": "ViT-L/16 (larger model, 304M params)"
    },

    # Self-supervised: DINOv2
    "dinov2-base": {
        "hf_name": "facebook/dinov2-base",
        "img_size": 518,  # DINOv2 optimal resolution
        "patch_size": 14,
        "base_lr": 5e-5,
        "head_lr": 2e-3,
        "batch_size": 8,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.85,
        "weight_decay": 0.04,
        "warmup_ratio": 0.1,
        "description": "DINOv2-B (self-supervised)"
    },
    "dinov2-large": {
        "hf_name": "facebook/dinov2-large",
        "img_size": 518,
        "patch_size": 14,
        "base_lr": 3e-5,
        "head_lr": 1e-3,
        "batch_size": 4,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.90,
        "weight_decay": 0.04,
        "warmup_ratio": 0.1,
        "description": "DINOv2-L (304M params)"
    },

    # DeiT: supervised training with a distillation token (grouped with the
    # supervised paradigm in the paper)
    "deit-base": {
        "hf_name": "facebook/deit-base-distilled-patch16-224",
        "img_size": 224,
        "patch_size": 16,
        "base_lr": 5e-5,
        "head_lr": 2e-3,
        "batch_size": 16,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.85,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "description": "DeiT-B (supervised, distillation token)"
    },

    # Vision-language: CLIP
    "clip-base": {
        "hf_name": "openai/clip-vit-base-patch16",
        "img_size": 224,
        "patch_size": 16,
        "base_lr": 5e-6,  # CLIP needs lower LR
        "head_lr": 1e-4,
        "batch_size": 16,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.85,
        "weight_decay": 0.02,
        "warmup_ratio": 0.1,
        "description": "CLIP-B (trained on image-text pairs)"
    },
    "clip-large": {
        "hf_name": "openai/clip-vit-large-patch14",
        "img_size": 224,
        "patch_size": 14,
        "base_lr": 3e-6,
        "head_lr": 5e-5,
        "batch_size": 8,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.90,
        "weight_decay": 0.02,
        "warmup_ratio": 0.1,
        "description": "CLIP-L (large vision-language model)"
    },

    # Smaller supervised baseline
    "vit-small-224": {
        "hf_name": "WinKawaks/vit-small-patch16-224",
        "img_size": 224,
        "patch_size": 16,
        "base_lr": 5e-5,
        "head_lr": 2e-3,
        "batch_size": 32,
        "epochs_head": 5,
        "epochs_partial": 5,
        "epochs_full": 5,
        "llrd_decay": 0.85,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "description": "ViT-S/16 (smaller, faster baseline - 22M params)"
    },
}

# NOTE: MAE models (facebook/vit-mae-base) are NOT compatible with AutoModelForImageClassification
# They are pre-trained for masked autoencoding (reconstruction), not classification.
# Use DINOv2 or DeiT for self-supervised alternatives instead.

# ============================================================
# 3) Configuration
# ============================================================
@dataclass
class Config:
    """Global configuration for experiments."""

    # Data paths
    BASE_PATH: str = "/content/webdesignprototypicality"
    CATEGORIES: Tuple[str, ...] = ("banks", "fashion", "homeware", "universities")

    # Models to evaluate (set which models to run)
    MODELS_TO_RUN: List[str] = field(default_factory=lambda: [
      "vit-small-224",
      "vit-base-224",
      "vit-base-384",
      "vit-large-384",
      "dinov2-base",
      "dinov2-large",
      "deit-base",
      "clip-base",
      "clip-large",
    ])

    # Training hyperparameters (defaults, overridden by model-specific)
    ACCUM_STEPS: int = 1
    MAX_GRAD_NORM: float = 1.0
    EARLY_STOP_PATIENCE: int = 3
    EARLY_STOP_KEY: str = "r"  # "rmse", "r", or "rho"

    # Data split ratios
    HOLDOUT_TEST_SIZE: float = 0.20
    VAL_SIZE: float = 0.15
    HOLDOUT_SEED: int = 42
    VAL_SEED: int = 202

    # Loss function
    LOSS: str = "mse"  # "mse", "mae", "huber", "logcosh", "npearson", "ccc", "mse+npearson"
    HUBER_DELTA: float = 0.5
    ALPHA_R: float = 0.1

    # Evaluation
    USE_TTA: bool = False  # Test-time augmentation (horizontal flip)
    BOOTSTRAP_N: int = 2000
    BOOTSTRAP_ALPHA: float = 0.05

    # Paths
    CACHE_DIR: str = "/content/cache_vit"
    CKPT_DIR: str = "/content/checkpoints"
    LOG_DIR: str = "/content/logs"
    RESULTS_DIR: str = "/content/drive/MyDrive/results/vit_benchmark/model_comparison"

    # DataLoader (set to 0 for Colab/Jupyter to avoid multiprocessing issues)
    NUM_WORKERS: int = 0
    PREFETCH_FACTOR: int = 2

    # Experiment tracking
    EXPERIMENT_NAME: str = f"vit_benchmark_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    SAVE_PREDICTIONS: bool = True
    SAVE_ATTENTION_MAPS: bool = False

    def __post_init__(self):
        # Mount Drive and unpack the dataset on first run
        from google.colab import drive
        drive.mount('/content/drive')
        if not os.path.exists(self.BASE_PATH):
            os.system(f'unzip -q /content/drive/MyDrive/datasets/webdesignprototypicality.zip -d {self.BASE_PATH}')

        for dir_path in [self.CACHE_DIR, self.CKPT_DIR, self.LOG_DIR, self.RESULTS_DIR]:
            os.makedirs(dir_path, exist_ok=True)

        # Create experiment subdirectory
        self.exp_dir = os.path.join(self.RESULTS_DIR, self.EXPERIMENT_NAME)
        os.makedirs(self.exp_dir, exist_ok=True)

        # Validate models
        invalid_models = [m for m in self.MODELS_TO_RUN if m not in MODEL_REGISTRY]
        if invalid_models:
            raise ValueError(f"Invalid models: {invalid_models}. Choose from: {list(MODEL_REGISTRY.keys())}")

CFG = Config()

# ============================================================
# 4) Logging Setup
# ============================================================
class ExperimentLogger:
    """Structured logging for experiments."""

    def __init__(self, log_dir: str, experiment_name: str):
        self.log_dir = log_dir
        self.log_file = os.path.join(log_dir, f"{experiment_name}.log")
        self.results = {}

    def log(self, message: str, level: str = "INFO"):
        """Log message to file and console."""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] [{level}] {message}"
        print(log_msg)
        with open(self.log_file, "a") as f:
            f.write(log_msg + "\n")

    def log_config(self, config: Dict[str, Any]):
        """Log configuration."""
        self.log("=" * 80)
        self.log("EXPERIMENT CONFIGURATION")
        self.log("=" * 80)
        for key, value in config.items():
            self.log(f"{key}: {value}")
        self.log("=" * 80)

    def log_model_info(self, model_name: str, model_config: Dict[str, Any]):
        """Log model-specific information."""
        self.log(f"\n{'='*80}")
        self.log(f"MODEL: {model_name}")
        self.log(f"{'='*80}")
        for key, value in model_config.items():
            self.log(f"  {key}: {value}")

    def log_results(self, model_name: str, results: Dict[str, Any]):
        """Log results for a model."""
        self.results[model_name] = results
        self.log(f"\nResults for {model_name}:")
        self.log(f"  Overall RMSE: {results['overall']['rmse']:.4f}")
        self.log(f"  Overall r: {results['overall']['r']:.4f}")
        self.log(f"  Overall ρ: {results['overall']['rho']:.4f}")
        self.log(f"  Training time: {results['training_time_hrs']:.2f} hrs")

    def save_final_results(self):
        """Save all results to JSON."""
        results_path = os.path.join(CFG.exp_dir, "all_results.json")
        with open(results_path, "w") as f:
            json.dump(self.results, f, indent=2)
        self.log(f"\nAll results saved to: {results_path}")

logger = ExperimentLogger(CFG.LOG_DIR, CFG.EXPERIMENT_NAME)
logger.log_config({
    "experiment_name": CFG.EXPERIMENT_NAME,
    "models_to_run": CFG.MODELS_TO_RUN,
    "dataset_path": CFG.BASE_PATH,
    "device": str(device),
    "cuda_available": use_cuda,
})

# ============================================================
# 5) Loss Functions
# ============================================================
def pearson_corr(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Compute Pearson correlation."""
    x = x - x.mean()
    y = y - y.mean()
    x = x / (x.std() + eps)
    y = y / (y.std() + eps)
    return (x * y).mean()

def ccc_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Concordance Correlation Coefficient loss."""
    x = pred.squeeze(1)
    y = target.squeeze(1)
    mean_x, mean_y = x.mean(), y.mean()
    vx = x.var(unbiased=False)
    vy = y.var(unbiased=False)
    cov = ((x - mean_x) * (y - mean_y)).mean()
    ccc = (2 * cov) / (vx + vy + (mean_x - mean_y).pow(2) + eps)
    return 1.0 - ccc

def log_cosh_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Log-cosh loss (smooth L1 variant)."""
    z = (pred - target).squeeze(1)
    return torch.mean(torch.log(torch.cosh(z + 1e-12)))

def huber_loss(pred: torch.Tensor, target: torch.Tensor, beta: float = 0.5) -> torch.Tensor:
    """Huber loss."""
    try:
        return F.smooth_l1_loss(pred, target, beta=beta)
    except TypeError:
        return F.smooth_l1_loss(pred, target)

def neg_pearson_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Negative Pearson correlation as loss."""
    return 1.0 - pearson_corr(pred.squeeze(1), target.squeeze(1))

def primary_loss_fn(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Primary loss function based on config."""
    if CFG.LOSS == "mse":
        return F.mse_loss(pred, target)
    elif CFG.LOSS == "mae":
        return F.l1_loss(pred, target)
    elif CFG.LOSS == "huber":
        return huber_loss(pred, target, beta=CFG.HUBER_DELTA)
    elif CFG.LOSS == "logcosh":
        return log_cosh_loss(pred, target)
    elif CFG.LOSS == "npearson":
        return neg_pearson_loss(pred, target)
    elif CFG.LOSS == "ccc":
        return ccc_loss(pred, target)
    elif CFG.LOSS == "mse+npearson":
        return F.mse_loss(pred, target) + CFG.ALPHA_R * neg_pearson_loss(pred, target)
    else:
        raise ValueError(f"Unknown loss: {CFG.LOSS}")

# ============================================================
# 6) Data Loading
# ============================================================
def _ratings_path(category: str) -> str:
    return os.path.join(CFG.BASE_PATH, category, f"ratings.avg.{category}.txt")

def _screenshot_dir(category: str) -> str:
    return os.path.join(CFG.BASE_PATH, category, "screenshots")

def load_webdesign_data() -> pd.DataFrame:
    """Load website aesthetic ratings dataset."""
    rows = []
    for cat in CFG.CATEGORIES:
        ratings_file = _ratings_path(cat)
        screenshot_dir = _screenshot_dir(cat)

        if not os.path.exists(ratings_file):
            logger.log(f"Missing ratings file: {ratings_file}", level="WARN")
            continue

        # Read ratings (TSV format: stimulusId, [other cols], AE)
        df = pd.read_csv(ratings_file, sep="\t")

        if not {"stimulusId", "AE"}.issubset(df.columns):
            raise RuntimeError(f"Expected columns 'stimulusId' and 'AE' in {ratings_file}")

        # Build screenshot paths
        df["category"] = cat
        df["screenshot_path"] = df["stimulusId"].apply(
            lambda s: os.path.join(screenshot_dir, s)
        )

        # Filter to existing files only
        df = df[df["screenshot_path"].apply(os.path.exists)]

        rows.append(df[["stimulusId", "category", "AE", "screenshot_path"]].copy())

    if not rows:
        raise RuntimeError("No data loaded. Check BASE_PATH and dataset structure.")

    df_all = pd.concat(rows, axis=0, ignore_index=True)
    df_all.rename(columns={"AE": "label"}, inplace=True)

    # Remove duplicates (by path)
    n_dupes = df_all.duplicated(["screenshot_path"]).sum()
    if n_dupes > 0:
        logger.log(f"Found {n_dupes} duplicate paths; keeping first occurrence", level="WARN")
        df_all = df_all.drop_duplicates(["screenshot_path"], keep="first").reset_index(drop=True)

    logger.log(f"Loaded {len(df_all)} samples")
    logger.log(f"Category distribution:\n{df_all['category'].value_counts()}")

    return df_all


# Letterbox preprocessing
def letterbox_square(img: Image.Image, out_size: int, pad_color: Tuple[int, int, int] = (128, 128, 128)) -> Image.Image:
    """Letterbox image to square with padding, then resize."""
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), pad_color)
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((out_size, out_size), Image.BICUBIC)

def cache_path_for(src_path: str, size: int) -> str:
    """Generate cache file path."""
    base = os.path.basename(src_path).replace(".jpg", f"_{size}.jpg")
    cat = os.path.basename(os.path.dirname(os.path.dirname(src_path)))
    return os.path.join(CFG.CACHE_DIR, f"{cat}_{base}")

def load_or_build_cached(path: str, size: int) -> Image.Image:
    """Load cached resized image or create if missing."""
    cp = cache_path_for(path, size)

    if os.path.exists(cp):
        try:
            return Image.open(cp).convert("RGB")
        except Exception:
            os.remove(cp)  # Corrupted cache

    # Build cache
    with Image.open(path) as im:
        im = im.convert("RGB")
        im = letterbox_square(im, size, (128, 128, 128))
        os.makedirs(os.path.dirname(cp), exist_ok=True)
        im.save(cp, format="JPEG", quality=95)

    return Image.open(cp).convert("RGB")

def to_tensor_normalized(img: Image.Image, mean: List[float], std: List[float]) -> torch.Tensor:
    """Convert PIL image to normalized tensor."""
    t = TF.to_tensor(img)
    mean_t = torch.tensor(mean).view(-1, 1, 1)
    std_t = torch.tensor(std).view(-1, 1, 1)
    return (t - mean_t) / std_t

# Dataset
class WebAestheticsDataset(Dataset):
    """Website aesthetics dataset."""

    def __init__(
        self,
        frame: pd.DataFrame,
        img_size: int,
        mean: List[float],
        std: List[float],
        return_meta: bool = False
    ):
        self.df = frame.reset_index(drop=True)
        self.size = img_size
        self.mean = mean
        self.std = std
        self.return_meta = return_meta

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        # Load and preprocess image
        img = load_or_build_cached(row["screenshot_path"], self.size)
        x = to_tensor_normalized(img, self.mean, self.std)

        # Label
        y = torch.tensor([float(row["label"])], dtype=torch.float32)

        out = {"pixel_values": x, "labels": y}

        if self.return_meta:
            out.update({
                "id": row["stimulusId"],
                "cat": row["category"],
                "path": row["screenshot_path"]
            })

        return out

class InferenceDataset(Dataset):
    """Dataset for inference (includes metadata)."""

    def __init__(self, frame: pd.DataFrame, img_size: int, mean: List[float], std: List[float]):
        self.df = frame.reset_index(drop=True)
        self.size = img_size
        self.mean = mean
        self.std = std

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        img = load_or_build_cached(row["screenshot_path"], self.size)
        x = to_tensor_normalized(img, self.mean, self.std)

        return {
            "pixel_values": x,
            "id": row["stimulusId"],
            "category": row["category"],
            "path": row["screenshot_path"],
            "row_idx": idx
        }

# Stratification helper
def stratify_key(df: pd.DataFrame, bins: int = 6) -> np.ndarray:
    """Create stratification key (category + label quantile)."""
    cats = df["category"].astype("category").cat.codes.values
    y = df["label"].values
    qs = np.quantile(y, np.linspace(0, 1, bins + 1))

    # Ensure quantiles are strictly increasing
    for i in range(1, len(qs)):
        if qs[i] <= qs[i-1]:
            qs[i] = qs[i-1] + 1e-6

    yb = np.digitize(y, qs[1:-1], right=True)
    return cats * (bins + 1) + yb

# ============================================================
# 7) Model Building & Training Utilities
# ============================================================
def build_model(model_key: str) -> Tuple[nn.Module, Dict[str, Any]]:
    """Build model from registry."""
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Model {model_key} not in registry")

    config = MODEL_REGISTRY[model_key]

    logger.log(f"Building model: {model_key}")
    logger.log(f"  HuggingFace: {config['hf_name']}")
    logger.log(f"  Image size: {config['img_size']}")

    # Load model
    model = AutoModelForImageClassification.from_pretrained(
        config['hf_name'],
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True
    )

    # Enable gradient checkpointing for memory efficiency
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    model.to(device)

    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    logger.log(f"  Total params: {n_params:,}")
    logger.log(f"  Trainable params: {n_trainable:,}")

    return model, config

def get_image_processor(hf_name: str) -> Tuple[List[float], List[float]]:
    """Get normalization statistics for model."""
    try:
        processor = AutoImageProcessor.from_pretrained(hf_name, use_fast=True)
        mean = processor.image_mean
        std = processor.image_std
    except Exception:
        # Fallback to ImageNet stats
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]

    return mean, std

def freeze_all(m: nn.Module, freeze: bool = True):
    """Freeze/unfreeze all parameters."""
    for p in m.parameters():
        p.requires_grad = not freeze

def get_encoder_blocks(m):
    """Locate the encoder and its transformer blocks. The attribute path differs
    per architecture (CLIP: vision_model.encoder.layers, ViT: vit.encoder.layer, ...)."""
    encoder = None
    if hasattr(m, "vision_model") and hasattr(m.vision_model, "encoder"):
        encoder = m.vision_model.encoder
    elif hasattr(m, "vit") and hasattr(m.vit, "encoder"):
        encoder = m.vit.encoder
    elif hasattr(m, "deit") and hasattr(m.deit, "encoder"):
        encoder = m.deit.encoder
    elif hasattr(m, "encoder"):
        encoder = m.encoder

    if encoder is None:
        return None, None, None

    for attr_name in ["layers", "layer", "blocks"]:
        if hasattr(encoder, attr_name):
            blocks = getattr(encoder, attr_name)
            if isinstance(blocks, (torch.nn.ModuleList, list)) and len(blocks) > 0:
                return encoder, list(blocks), attr_name

    return encoder, None, None

def get_head_param_names(m: nn.Module) -> List[str]:
    """Get classification head parameter names for different architectures."""
    head_patterns = [
        "classifier",           # Standard ViT, DINOv2
        "head",                 # DeiT, some custom models
        "fc",                   # ResNet-style
        "score",                # Some models
    ]

    head_names = []
    for name, _ in m.named_parameters():
        for pattern in head_patterns:
            # Check if name starts with pattern or contains it as a top-level component
            if name.startswith(pattern) or name.startswith(f"{pattern}."):
                head_names.append(name)
                break

    return head_names

def unfreeze_head(m: nn.Module):
    """Unfreeze only classification head (handles different naming conventions)."""
    freeze_all(m, freeze=True)

    head_names = get_head_param_names(m)

    if not head_names:
        # Fallback: try to find the last linear layer
        logger.log("  Warning: No standard head found, searching for last linear layer...", level="WARN")
        linear_layers = [(n, p) for n, p in m.named_parameters() if 'weight' in n and len(p.shape) == 2]
        if linear_layers:
            # Get the last few layers
            for name, param in linear_layers[-2:]:  # Unfreeze last 2 linear layers
                param.requires_grad = True
                logger.log(f"    Unfreezing: {name}")
        else:
            raise RuntimeError("Could not find classification head to unfreeze!")
    else:
        for name, p in m.named_parameters():
            if name in head_names:
                p.requires_grad = True

        logger.log(f"  Unfroze head with {len(head_names)} parameter groups")

    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    logger.log(f"  Trainable params: {n_trainable:,}")

def unfreeze_last_k_blocks(m, k=4):
    """Unfreeze the regression head, the top-level norm layers, and the last k encoder blocks."""
    freeze_all(m, freeze=True)

    # 1. Unfreeze head
    head_names = get_head_param_names(m)
    for name, p in m.named_parameters():
        if name in head_names:
            p.requires_grad = True

    # 2. Unfreeze top-level norms
    for n, p in m.named_parameters():
        if p.requires_grad:
            continue
        if ("layernorm" in n.lower() or "pooler" in n.lower() or "norm" in n.lower()):
            if not any(x in n.lower() for x in ["encoder.layer", "encoder.layers", "encoder.blocks"]):
                p.requires_grad = True

    # 3. Find encoder and blocks (CLIP compatible)
    encoder = None
    blocks = None
    blocks_attr = None

    # Try to find encoder
    if hasattr(m, "vision_model") and hasattr(m.vision_model, "encoder"):
        encoder = m.vision_model.encoder
    elif hasattr(m, "vit") and hasattr(m.vit, "encoder"):
        encoder = m.vit.encoder
    elif hasattr(m, "deit") and hasattr(m.deit, "encoder"):
        encoder = m.deit.encoder
    elif hasattr(m, "encoder"):
        encoder = m.encoder

    # Try to find blocks with CLIP's naming
    if encoder is not None:
        for attr in ["layers", "layer", "blocks"]:  # Try "layers" first for CLIP
            if hasattr(encoder, attr):
                blocks = getattr(encoder, attr)
                if isinstance(blocks, (nn.ModuleList, list)) and len(blocks) > 0:
                    blocks_attr = attr
                    break

    # Unfreeze last k blocks
    if blocks is not None:
        blocks = list(blocks)
        depth = len(blocks)
        for i in range(max(0, depth - k), depth):
            for p in blocks[i].parameters():
                p.requires_grad = True
        logger.log(f"  Unfroze last {k}/{depth} encoder blocks (via {blocks_attr})")
    else:
        logger.log("  Warning: Could not find encoder blocks, unfreezing all", level="WARN")
        freeze_all(m, freeze=False)

    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    logger.log(f"  Total trainable params: {n_trainable:,}")

def unfreeze_all(m: nn.Module):
    """Unfreeze all parameters."""
    freeze_all(m, freeze=False)
    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    logger.log(f"  All params trainable: {n_trainable:,}")

def make_param_groups_llrd(m, base_lr, head_lr, decay):
    """Optimizer parameter groups with layer-wise learning rate decay:
    block l gets base_lr * decay**(depth - 1 - l), the head keeps its own rate."""
    groups = []
    assigned_params = set()

    def param_id(p):
        return id(p)

    # 1. Head
    head_names = get_head_param_names(m)
    head_params = [p for n, p in m.named_parameters() if n in head_names and p.requires_grad]
    if head_params:
        groups.append({"params": head_params, "lr": head_lr})
        for p in head_params:
            assigned_params.add(param_id(p))

    # 2. Find encoder blocks
    encoder = None
    blocks = None

    if hasattr(m, "vision_model") and hasattr(m.vision_model, "encoder"):
        encoder = m.vision_model.encoder
    elif hasattr(m, "vit") and hasattr(m.vit, "encoder"):
        encoder = m.vit.encoder
    elif hasattr(m, "deit") and hasattr(m.deit, "encoder"):
        encoder = m.deit.encoder
    elif hasattr(m, "encoder"):
        encoder = m.encoder

    if encoder is not None:
        for attr in ["layers", "layer", "blocks"]:
            if hasattr(encoder, attr):
                blocks = getattr(encoder, attr)
                if isinstance(blocks, (nn.ModuleList, list)) and len(blocks) > 0:
                    break

    # Collect encoder block params
    encoder_block_params = set()
    if blocks is not None:
        for block in blocks:
            for p in block.parameters():
                if p.requires_grad:
                    encoder_block_params.add(param_id(p))

    # 3. LayerNorm & Pooler
    ln_pool_params = []
    for n, p in m.named_parameters():
        if p.requires_grad and param_id(p) not in assigned_params:
            if any(x in n.lower() for x in ["layernorm", "pooler", "norm"]):
                if param_id(p) not in encoder_block_params:
                    ln_pool_params.append(p)
                    assigned_params.add(param_id(p))

    if ln_pool_params:
        groups.append({"params": ln_pool_params, "lr": base_lr})

    # 4. Encoder blocks with LLRD
    if blocks is not None:
        blocks = list(blocks)
        depth = len(blocks)

        for i in range(depth - 1, -1, -1):
            block_params = []
            for p in blocks[i].parameters():
                if p.requires_grad and param_id(p) not in assigned_params:
                    block_params.append(p)
                    assigned_params.add(param_id(p))

            if block_params:
                scale = decay ** (depth - 1 - i)
                groups.append({"params": block_params, "lr": base_lr * scale})

    # 5. Remaining params
    remaining = [p for p in m.parameters() if p.requires_grad and param_id(p) not in assigned_params]
    if remaining:
        groups.append({"params": remaining, "lr": base_lr})

    return groups

# ============================================================
# 8) Training & Evaluation
# ============================================================
def evaluate(model: nn.Module, loader: DataLoader) -> Tuple:
    """Evaluate model on a dataset."""
    model.eval()
    losses, y_true, y_pred = [], [], []

    with torch.no_grad(), amp.autocast('cuda', enabled=use_cuda):
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            x = batch["pixel_values"].to(device, non_blocking=True)
            y = batch["labels"].to(device, non_blocking=True)

            out = model(x).logits
            loss = primary_loss_fn(out, y)

            losses.append(loss.item())
            y_true.append(y.squeeze(1).cpu().float().numpy())
            y_pred.append(out.squeeze(1).cpu().float().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    val_loss = float(np.mean(losses))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    r = float(pearsonr(y_pred, y_true)[0]) if len(y_true) > 1 else float("nan")
    rho = float(spearmanr(y_pred, y_true)[0]) if len(y_true) > 1 else float("nan")

    return val_loss, rmse, r, rho, y_true, y_pred

def get_early_stop_score(val_loss: float, rmse: float, r: float, rho: float) -> float:
    """Get score for early stopping (lower is better)."""
    if CFG.EARLY_STOP_KEY == "rmse":
        return rmse
    elif CFG.EARLY_STOP_KEY == "val_loss":
        return val_loss
    elif CFG.EARLY_STOP_KEY == "r":
        return -r
    elif CFG.EARLY_STOP_KEY == "rho":
        return -rho
    return rmse

def train_phase(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    phase_name: str
) -> Dict[str, Any]:
    """Train model for one phase."""
    scaler = amp.GradScaler(enabled=use_cuda)
    best = {
        "score": float("inf"),
        "state_dict": None,
        "epoch": 0,
        "val": {}
    }

    patience = 0
    step = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_losses = []

        pbar = tqdm(train_loader, desc=f"{phase_name} Epoch {epoch}/{epochs}", leave=False)

        for batch in pbar:
            x = batch["pixel_values"].to(device, non_blocking=True)
            y = batch["labels"].to(device, non_blocking=True)

            with amp.autocast('cuda', enabled=use_cuda):
                out = model(x).logits
                loss = primary_loss_fn(out, y) / CFG.ACCUM_STEPS

            scaler.scale(loss).backward()

            if (step + 1) % CFG.ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            step += 1
            running_losses.append(loss.item() * CFG.ACCUM_STEPS)

            if len(running_losses) % 10 == 0:
                pbar.set_postfix(loss=np.mean(running_losses[-10:]))

        # Validation
        val_loss, rmse, r, rho, _, _ = evaluate(model, val_loader)
        score = get_early_stop_score(val_loss, rmse, r, rho)

        logger.log(
            f"{phase_name} Epoch {epoch}: "
            f"val_loss={val_loss:.4f} RMSE={rmse:.4f} r={r:.3f} ρ={rho:.3f}"
        )

        # Check for improvement
        if score < best["score"] - 1e-6:
            best.update({
                "score": score,
                "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "epoch": epoch,
                "val": {"val_loss": val_loss, "rmse": rmse, "r": r, "rho": rho}
            })
            patience = 0
        else:
            patience += 1
            if patience >= CFG.EARLY_STOP_PATIENCE:
                logger.log(f"Early stopping at epoch {epoch}. Best: epoch {best['epoch']}")
                break

    # Load best weights
    if best["state_dict"] is not None:
        model.load_state_dict(best["state_dict"], strict=True)

        # Free the cached state dict
        del best["state_dict"]
        torch.cuda.empty_cache()

    return best

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict[str, Any]
) -> Dict[str, Any]:
    """Full 3-phase training pipeline."""
    results = {}

    # Phase 1: Head only
    logger.log("\n" + "="*60)
    logger.log("Phase 1: Head Only")
    logger.log("="*60)
    unfreeze_head(model)

    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=config['head_lr'],
        weight_decay=config['weight_decay']
    )
    steps = config['epochs_head'] * math.ceil(len(train_loader) / CFG.ACCUM_STEPS)
    sch = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(config['warmup_ratio'] * steps),
        num_training_steps=steps
    )

    best_phase1 = train_phase(
        model, train_loader, val_loader,
        config['epochs_head'], opt, sch, "Phase1"
    )
    results['phase1'] = best_phase1['val']

    # Clean up Phase 1
    del opt, sch
    torch.cuda.empty_cache()

    # Phase 2: Last K blocks
    logger.log("\n" + "="*60)
    logger.log("Phase 2: Last K Blocks + LLRD")
    logger.log("="*60)
    unfreeze_last_k_blocks(model, k=4)

    param_groups = make_param_groups_llrd(
        model,
        base_lr=config['base_lr'],
        head_lr=config['head_lr'],
        decay=config['llrd_decay']
    )
    opt = torch.optim.AdamW(param_groups, weight_decay=config['weight_decay'])
    steps = config['epochs_partial'] * math.ceil(len(train_loader) / CFG.ACCUM_STEPS)
    sch = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(config['warmup_ratio'] * steps),
        num_training_steps=steps
    )

    best_phase2 = train_phase(
        model, train_loader, val_loader,
        config['epochs_partial'], opt, sch, "Phase2"
    )
    results['phase2'] = best_phase2['val']

    # Clean up Phase 2
    del opt, sch, param_groups
    torch.cuda.empty_cache()

    # Phase 3: Full fine-tuning
    logger.log("\n" + "="*60)
    logger.log("Phase 3: Full Fine-tuning")
    logger.log("="*60)
    unfreeze_all(model)

    opt = torch.optim.AdamW(
        [{"params": [p for p in model.parameters() if p.requires_grad],
          "lr": config['base_lr']}],
        weight_decay=config['weight_decay']
    )
    steps = config['epochs_full'] * math.ceil(len(train_loader) / CFG.ACCUM_STEPS)
    sch = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(config['warmup_ratio'] * steps),
        num_training_steps=steps
    )

    best_phase3 = train_phase(
        model, train_loader, val_loader,
        config['epochs_full'], opt, sch, "Phase3"
    )
    results['phase3'] = best_phase3['val']

    # Clean up Phase 3
    del opt, sch
    torch.cuda.empty_cache()

    return results

# ============================================================
# 9) Memory Management
# ============================================================
def print_memory_stats(prefix=""):
    """Print current memory usage."""
    import psutil
    import os

    # RAM
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / 1024**3

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        logger.log(f"{prefix}RAM: {ram_gb:.2f}GB | GPU: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    else:
        logger.log(f"{prefix}RAM: {ram_gb:.2f}GB")

def aggressive_memory_cleanup():
    """Aggressively clean up memory between model runs."""
    import gc

    # Collect garbage multiple times (sometimes needed)
    for _ in range(3):
        gc.collect()

    # Clear CUDA cache if available
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    # Clear matplotlib figures
    try:
        plt.close('all')
    except:
        pass

    # Print final memory state
    print_memory_stats("  After cleanup: ")

# ============================================================
# 10) Inference & Metrics
# ============================================================
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, use_tta: bool = True) -> Tuple:
    """Run inference with optional TTA."""
    model.eval()
    preds, metas = [], []

    with amp.autocast('cuda', enabled=use_cuda):
        for batch in tqdm(loader, desc="Inference", leave=False):
            x = batch["pixel_values"].to(device, non_blocking=True)

            if use_tta:
                # Average predictions from original and horizontally flipped
                out1 = model(x).logits
                out2 = model(torch.flip(x, dims=[-1])).logits
                out = (out1 + out2) / 2
            else:
                out = model(x).logits

            preds.append(out.squeeze(1).cpu().numpy())

            metas.append(pd.DataFrame({
                "row_idx": batch["row_idx"],
                "stimulusId": batch["id"],
                "category": batch["category"],
                "path": batch["path"]
            }))

    return np.concatenate(preds), pd.concat(metas, ignore_index=True)

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute evaluation metrics."""
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    r = float(pearsonr(y_pred, y_true)[0]) if len(y_true) > 1 else float("nan")
    rho = float(spearmanr(y_pred, y_true)[0]) if len(y_true) > 1 else float("nan")

    return {"rmse": rmse, "mae": mae, "r": r, "rho": rho}

def bootstrap_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    metric_name: str,
    n_boot: int = 2000,
    alpha: float = 0.05,
    seed: int = 42
) -> Tuple[float, float]:
    """Compute bootstrap confidence interval."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, yp = y_true[idx], y_pred[idx]

        if metric_name == "rmse":
            val = np.sqrt(np.mean((yp - yt) ** 2))
        elif metric_name == "r":
            val = pearsonr(yp, yt)[0]
        elif metric_name == "rho":
            val = spearmanr(yp, yt)[0]
        else:
            raise ValueError(f"Unknown metric: {metric_name}")

        vals.append(val)

    lo = float(np.percentile(vals, 100 * (alpha / 2)))
    hi = float(np.percentile(vals, 100 * (1 - alpha / 2)))

    return lo, hi

def evaluate_per_category(
    df: pd.DataFrame,
    y_true: np.ndarray,
    y_pred: np.ndarray
) -> Dict[str, Dict[str, float]]:
    """Compute per-category metrics."""
    df = df.reset_index(drop=True)
    results = {}

    for cat in sorted(df["category"].unique()):
        mask = (df["category"] == cat).values
        if not mask.any():
            continue

        yt = y_true[mask]
        yp = y_pred[mask]

        results[cat] = compute_metrics(yt, yp)

    return results

# ============================================================
# 11) Main Experiment Loop
# ============================================================
def run_experiment():
    """Main experiment loop for all models."""
    logger.log("\n" + "="*80)
    logger.log("STARTING BENCHMARK EXPERIMENT")
    logger.log("="*80)

    # Load data
    logger.log("\nLoading dataset...")
    df_all = load_webdesign_data()

    # Create stratified splits
    logger.log("\nCreating stratified splits...")
    y_strat = stratify_key(df_all)

    # Hold-out split
    sss = StratifiedShuffleSplit(
        n_splits=1,
        test_size=CFG.HOLDOUT_TEST_SIZE,
        random_state=CFG.HOLDOUT_SEED
    )
    tr_idx, te_idx = next(sss.split(np.zeros(len(df_all)), y_strat))
    df_train_all = df_all.iloc[tr_idx].reset_index(drop=True)
    df_test = df_all.iloc[te_idx].reset_index(drop=True)

    # Validation split
    y_strat_train = stratify_key(df_train_all)
    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=CFG.VAL_SIZE,
        random_state=CFG.VAL_SEED
    )
    tr_i, va_i = next(sss_val.split(np.zeros(len(df_train_all)), y_strat_train))
    df_train = df_train_all.iloc[tr_i].reset_index(drop=True)
    df_val = df_train_all.iloc[va_i].reset_index(drop=True)

    logger.log(f"  Train: {len(df_train)} samples")
    logger.log(f"  Val: {len(df_val)} samples")
    logger.log(f"  Test: {len(df_test)} samples")

    # Verify no leakage
    train_paths = set(df_train["screenshot_path"])
    val_paths = set(df_val["screenshot_path"])
    test_paths = set(df_test["screenshot_path"])

    assert len(train_paths & test_paths) == 0, "Train-test leakage detected!"
    assert len(val_paths & test_paths) == 0, "Val-test leakage detected!"
    logger.log("  ✓ No data leakage detected")

    # Distribution statistics
    def stats(df):
        y = df["label"].values
        return f"mean={y.mean():.3f} std={y.std():.3f} min={y.min():.3f} max={y.max():.3f}"

    logger.log(f"\n  Train labels: {stats(df_train)}")
    logger.log(f"  Val labels: {stats(df_val)}")
    logger.log(f"  Test labels: {stats(df_test)}")

    # Run experiments for each model
    all_results = {}

    for model_key in CFG.MODELS_TO_RUN:
        logger.log("\n" + "="*80)
        logger.log(f"EVALUATING MODEL: {model_key}")
        logger.log("="*80)

        # Clean up before starting new model
        aggressive_memory_cleanup()
        print_memory_stats("Before model load: ")

        start_time = time.time()

        # Build model
        model, model_config = build_model(model_key)
        logger.log_model_info(model_key, model_config)

        # Get normalization stats
        mean, std = get_image_processor(model_config['hf_name'])
        logger.log(f"Normalization: mean={mean}, std={std}")

        # Create dataloaders
        train_ds = WebAestheticsDataset(df_train, model_config['img_size'], mean, std, return_meta=False)
        val_ds = WebAestheticsDataset(df_val, model_config['img_size'], mean, std, return_meta=True)
        test_ds = InferenceDataset(df_test, model_config['img_size'], mean, std)

        train_loader = DataLoader(
            train_ds,
            batch_size=model_config['batch_size'],
            shuffle=True,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=pin_memory,
            drop_last=True,
            prefetch_factor=CFG.PREFETCH_FACTOR if CFG.NUM_WORKERS > 0 else None,
            persistent_workers=False  # Better for notebooks
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=model_config['batch_size'] * 2,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=pin_memory,
            prefetch_factor=CFG.PREFETCH_FACTOR if CFG.NUM_WORKERS > 0 else None,
            persistent_workers=False
        )

        test_loader = DataLoader(
            test_ds,
            batch_size=model_config['batch_size'] * 2,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=pin_memory,
            prefetch_factor=CFG.PREFETCH_FACTOR if CFG.NUM_WORKERS > 0 else None,
            persistent_workers=False
        )

        # Train model
        training_results = train_model(model, train_loader, val_loader, model_config)

        # Memory check after training
        print_memory_stats("After training: ")

        # Final validation
        logger.log("\n" + "-"*60)
        logger.log("Final Validation")
        logger.log("-"*60)
        val_loss, rmse, r, rho, _, _ = evaluate(model, val_loader)
        logger.log(f"Val: RMSE={rmse:.4f} r={r:.4f} ρ={rho:.4f}")

        # Test set evaluation
        logger.log("\n" + "-"*60)
        logger.log("Test Set Evaluation")
        logger.log("-"*60)

        y_pred, meta = predict(model, test_loader, use_tta=CFG.USE_TTA)

        # Align predictions with test dataframe
        eval_df = meta.merge(
            df_test[["stimulusId", "category", "label"]].rename(columns={"label": "y_true"}),
            on=["stimulusId", "category"],
            how="left"
        )
        eval_df = eval_df.sort_values("row_idx").reset_index(drop=True)
        eval_df["y_pred"] = y_pred

        y_true = eval_df["y_true"].values
        y_pred = eval_df["y_pred"].values

        # Overall metrics
        overall_metrics = compute_metrics(y_true, y_pred)
        logger.log(f"Test Overall: RMSE={overall_metrics['rmse']:.4f} r={overall_metrics['r']:.4f} ρ={overall_metrics['rho']:.4f}")

        # Per-category metrics
        per_cat_metrics = evaluate_per_category(eval_df, y_true, y_pred)
        logger.log("\nPer-category results:")
        for cat, metrics in per_cat_metrics.items():
            logger.log(f"  {cat:>12}: RMSE={metrics['rmse']:.4f} r={metrics['r']:.3f} ρ={metrics['rho']:.3f}")

        # Bootstrap CIs
        logger.log("\nBootstrap 95% CIs:")
        rmse_ci = bootstrap_ci(y_true, y_pred, "rmse", n_boot=CFG.BOOTSTRAP_N, alpha=CFG.BOOTSTRAP_ALPHA)
        r_ci = bootstrap_ci(y_true, y_pred, "r", n_boot=CFG.BOOTSTRAP_N, alpha=CFG.BOOTSTRAP_ALPHA)
        rho_ci = bootstrap_ci(y_true, y_pred, "rho", n_boot=CFG.BOOTSTRAP_N, alpha=CFG.BOOTSTRAP_ALPHA)

        logger.log(f"  RMSE: [{rmse_ci[0]:.4f}, {rmse_ci[1]:.4f}]")
        logger.log(f"  r: [{r_ci[0]:.3f}, {r_ci[1]:.3f}]")
        logger.log(f"  ρ: [{rho_ci[0]:.3f}, {rho_ci[1]:.3f}]")

        # Save model
        ckpt_path = os.path.join(CFG.CKPT_DIR, f"{model_key}_final.pt")
        torch.save(model.state_dict(), ckpt_path)
        logger.log(f"\nSaved checkpoint: {ckpt_path}")

        # Save predictions
        if CFG.SAVE_PREDICTIONS:
            pred_path = os.path.join(CFG.exp_dir, f"{model_key}_predictions.csv")
            eval_df.to_csv(pred_path, index=False)
            logger.log(f"Saved predictions: {pred_path}")

        # Compute training time
        training_time = time.time() - start_time
        training_time_hrs = training_time / 3600

        # Store results
        all_results[model_key] = {
            "overall": overall_metrics,
            "per_category": per_cat_metrics,
            "confidence_intervals": {
                "rmse": {"lower": rmse_ci[0], "upper": rmse_ci[1]},
                "r": {"lower": r_ci[0], "upper": r_ci[1]},
                "rho": {"lower": rho_ci[0], "upper": rho_ci[1]},
            },
            "training_phases": training_results,
            "training_time_hrs": training_time_hrs,
            "n_params": sum(p.numel() for p in model.parameters()),
            "config": model_config
        }

        logger.log_results(model_key, all_results[model_key])

        # ============================================================
        # AGGRESSIVE MEMORY CLEANUP
        # ============================================================
        logger.log("\n" + "-"*60)
        logger.log("Cleaning up memory...")
        logger.log("-"*60)

        # 1. Delete dataloaders and datasets
        del train_loader, val_loader, test_loader
        del train_ds, val_ds, test_ds

        # 2. Delete model and related objects
        del model, model_config, mean, std

        # 3. Delete evaluation objects
        del y_pred, meta, eval_df, y_true
        del overall_metrics, per_cat_metrics
        del training_results

        # 4. Clear any cached computations
        if 'best_phase1' in locals():
            del best_phase1
        if 'best_phase2' in locals():
            del best_phase2
        if 'best_phase3' in locals():
            del best_phase3

        # 5. Call aggressive cleanup
        aggressive_memory_cleanup()

        logger.log(f"\nCompleted {model_key} in {training_time_hrs:.2f} hours")
        logger.log("="*80 + "\n")

    # Save final comparison
    logger.save_final_results()

    # Create comparison table
    comparison_df = create_comparison_table(all_results)
    comparison_path = os.path.join(CFG.exp_dir, "model_comparison.csv")
    comparison_df.to_csv(comparison_path, index=True)

    logger.log("\n" + "="*80)
    logger.log("FINAL COMPARISON TABLE")
    logger.log("="*80)
    logger.log("\n" + comparison_df.to_string())
    logger.log(f"\nComparison table saved: {comparison_path}")

    # Generate plots
    generate_comparison_plots(all_results, CFG.exp_dir)

    logger.log("\n" + "="*80)
    logger.log("EXPERIMENT COMPLETE")
    logger.log("="*80)
    logger.log(f"Results directory: {CFG.exp_dir}")

def create_comparison_table(results: Dict) -> pd.DataFrame:
    """Create comparison table from results."""
    rows = []

    for model_key, res in results.items():
        row = {
            "Model": model_key,
            "Params (M)": res["n_params"] / 1e6,
            "Image Size": res["config"]["img_size"],
            "RMSE": res["overall"]["rmse"],
            "r": res["overall"]["r"],
            "ρ": res["overall"]["rho"],
            "MAE": res["overall"]["mae"],
            "Training (hrs)": res["training_time_hrs"],
        }

        # Add per-category r
        for cat, metrics in res["per_category"].items():
            row[f"{cat}_r"] = metrics["r"]

        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values("r", ascending=False).reset_index(drop=True)

    return df

def generate_comparison_plots(results: Dict, output_dir: str):
    """Generate comparison visualizations."""
    # Overall performance
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    models = list(results.keys())
    rmses = [results[m]["overall"]["rmse"] for m in models]
    rs = [results[m]["overall"]["r"] for m in models]
    rhos = [results[m]["overall"]["rho"] for m in models]

    axes[0].barh(models, rmses)
    axes[0].set_xlabel("RMSE (lower is better)")
    axes[0].set_title("RMSE Comparison")
    axes[0].invert_xaxis()

    axes[1].barh(models, rs)
    axes[1].set_xlabel("Pearson r (higher is better)")
    axes[1].set_title("Pearson Correlation")

    axes[2].barh(models, rhos)
    axes[2].set_xlabel("Spearman ρ (higher is better)")
    axes[2].set_title("Spearman Correlation")

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "overall_comparison.png"), dpi=300, bbox_inches="tight")
    plt.close()

    logger.log(f"Saved plot: {os.path.join(output_dir, 'overall_comparison.png')}")

# Run the benchmark, then release the Colab runtime
run_experiment()

try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    pass


Using device: cuda
GPU: Tesla T4
CUDA Version: 12.6
Mounted at /content/drive
[2026-01-27 12:07:24] [INFO] ================================================================================
[2026-01-27 12:07:24] [INFO] EXPERIMENT CONFIGURATION
[2026-01-27 12:07:24] [INFO] ================================================================================
[2026-01-27 12:07:24] [INFO] experiment_name: vit_benchmark_20260127_120510
[2026-01-27 12:07:24] [INFO] models_to_run: ['vit-small-224', 'vit-base-224', 'vit-base-384', 'vit-large-384', 'dinov2-base', 'dinov2-large', 'deit-base', 'clip-base', 'clip-large']
[2026-01-27 12:07:24] [INFO] dataset_path: /content/webdesignprototypicality
[2026-01-27 12:07:24] [INFO] device: cuda
[2026-01-27 12:07:24] [INFO] cuda_available: True
[2026-01-27 12:07:24] [INFO] ================================================================================
[2026-01-27 12:07:24] [INFO] 
[2026-01-27 12:07:24] [INFO] STARTING BENCHMARK EXPERIMENT
[2026-01-27 12:07:24] 

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at WinKawaks/vit-small-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 384]) in the checkpoint and torch.Size([1, 384]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[2026-01-27 12:07:40] [INFO]   Total params: 21,666,049
[2026-01-27 12:07:40] [INFO]   Trainable params: 21,666,049
[2026-01-27 12:07:40] [INFO] 
[2026-01-27 12:07:40] [INFO] MODEL: vit-small-224
[2026-01-27 12:07:40] [INFO] ================================================================================
[2026-01-27 12:07:40] [INFO]   hf_name: WinKawaks/vit-small-patch16-224
[2026-01-27 12:07:40] [INFO]   img_size: 224
[2026-01-27 12:07:40] [INFO]   patch_size: 16
[2026-01-27 12:07:40] [INFO]   base_lr: 5e-05
[2026-01-27 12:07:40] [INFO]   head_lr: 0.002
[2026-01-27 12:07:40] [INFO]   batch_size: 32
[2026-01-27 12:07:40] [INFO]   epochs_head: 5
[2026-01-27 12:07:40] [INFO]   epochs_partial: 5
[2026-01-27 12:07:40] [INFO]   epochs_full: 5
[2026-01-27 12:07:40] [INFO]   llrd_decay: 0.85
[2026-01-27 12:07:40] [INFO]   weight_decay: 0.05
[2026-01-27 12:07:40] [INFO]   warmup_ratio: 0.1
[2026-01-27 12:07:40] [INFO]   description: ViT-S/16 (smaller, faster baseline - 22M params)


...

[2026-01-27 17:02:02] [INFO] Phase3 Epoch 5: val_loss=0.2762 RMSE=0.5242 r=0.633 ρ=0.594
[2026-01-27 17:02:02] [INFO] After training: RAM: 5.14GB | GPU: 1.15GB allocated, 1.15GB reserved
[2026-01-27 17:02:02] [INFO] 
------------------------------------------------------------
[2026-01-27 17:02:02] [INFO] Final Validation
[2026-01-27 17:02:02] [INFO] ------------------------------------------------------------


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

[2026-01-27 17:02:07] [INFO] Val: RMSE=0.5247 r=0.6338 ρ=0.5947
[2026-01-27 17:02:07] [INFO] 
------------------------------------------------------------
[2026-01-27 17:02:07] [INFO] Test Set Evaluation
[2026-01-27 17:02:07] [INFO] ------------------------------------------------------------


Inference:   0%|          | 0/40 [00:00<?, ?it/s]

[2026-01-27 17:02:15] [INFO] Test Overall: RMSE=0.5126 r=0.6642 ρ=0.6252
[2026-01-27 17:02:15] [INFO] 
Per-category results:
[2026-01-27 17:02:15] [INFO]          banks: RMSE=0.5095 r=0.680 ρ=0.617
[2026-01-27 17:02:15] [INFO]        fashion: RMSE=0.5824 r=0.408 ρ=0.385
[2026-01-27 17:02:15] [INFO]       homeware: RMSE=0.4832 r=0.684 ρ=0.655
[2026-01-27 17:02:15] [INFO]   universities: RMSE=0.4898 r=0.733 ρ=0.707
[2026-01-27 17:02:15] [INFO] 
Bootstrap 95% CIs:
[2026-01-27 17:02:17] [INFO]   RMSE: [0.4760, 0.5518]
[2026-01-27 17:02:17] [INFO]   r: [0.612, 0.708]
[2026-01-27 17:02:17] [INFO]   ρ: [0.571, 0.673]
[2026-01-27 17:02:24] [INFO] 
Saved checkpoint: /content/checkpoints/clip-large_final.pt
[2026-01-27 17:02:24] [INFO] Saved predictions: /content/drive/MyDrive/results/vit_benchmark/model_comparison/vit_benchmark_20260127_120510/clip-large_predictions.csv
[2026-01-27 17:02:24] [INFO] 
Results for clip-large:
[2026-01-27 17:02:24] [INFO]   Overall RMSE: 0.5126
[2026-01-27 17:02:24